### Problem Statement

A leading retail company wants to better understand its customers' shopping behavior in order to improve sales, customer satisfaction, and long-term loyalty.

The management team has noticed changes in purchasing patterns across demographics, product categories, and sales channels.

They are particularly interested in uncovering which factors, such as discounts, reviews, seasons, or payment preferences, drive consumer decisions and repeat purchases.


**You are tasked with analyzing the company's consumer behavior dataset to answer the following overarching business question:**

*"How can the company leverage consumer shopping data to identify trends, improve customer engagement, and optimize marketing and product strategies?"*

## Analysis Approach

- Load and explore the dataset to understand structure and variables  
- Clean the data by handling missing values and removing redundant columns  
- Standardize column names and engineer new analytical features  
- Load the cleaned dataset into SQLite for querying  
- Use SQL to analyze customer behavior, spending patterns, and subscriptions  
- Visualize insights with an interactive Power BI dashboard


In [1]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')


df = pd.read_csv('/content/drive/MyDrive/Data Analytics Project/customer_shopping_behavior.csv')

df.head()





Mounted at /content/drive


,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


### Inspect Dataset Structure

Identifying missing values and verifying each column has the expected data type.

In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

### Generate Summary Statistics

Visualizing overall distribution of the dataset.

In [5]:
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


### Identify Missing Values

Checking which columns contain null values.

In [6]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


### Fill Missing Review Ratings

Filling null values with the median review rating of each category to preserve category-level rating patterns instead of using a single global median.

In [7]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

### Verify Missing Values

Testing to see if null values were handled.

In [8]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


### Standardize Column Names

Standardizing column names by converting them to snake_case.

Consistent naming makes the dataset easier to work with in both Python and SQL queries.

In [9]:
df.columns = df.columns.str.replace(' ', '_')
df.columns = df.columns.str.lower()
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})
df.head()



,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


### Create Age Group Feature

Creating a new column called `age_group` that categorizes customers into age segments based on their age.  

Grouping customers this way allows us to analyze purchasing behavior across different demographic groups.

In [10]:
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)


### Validate Age Group Assignment

Displaying several rows from the `age` and `age_group` columns to confirm that customers were categorized correctly.


In [11]:
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


### Convert Purchase Frequency to Numeric Values

Converting categorical purchase frequency values such as Weekly, Fortnightly, and Monthly into numerical values representing the number of days between purchases.

This transformation allows us to perform quantitative analysis on purchasing frequency.

In [18]:
frequency_mapping = {'Fortnightly': 14,
                     'Weekly': 7,
                     'Monthly': 30,
                     'Quarterly': 90,
                     'Bi-Weekly': 14,
                     'Annually': 365,
                     'Every 3 Months': 90}


df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

### Confirm Frequency Conversion

Comparing the original purchase frequency column with the new numeric representation to ensure the conversion was applied correctly.


In [19]:
df[['purchase_frequency_days', 'frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


### Examine Discount and Promotion Variables

Inspecting the `discount_applied` and `promo_code_used` columns to determine whether both variables provide unique information or represent the same concept.


In [20]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


### Check for Redundant Data

Analyzing whether the promotion code column duplicates information already captured in the discount column.

In [ ]:
(df['discount_applied'] == df['promo_code_used']).all()


np.True_

### Remove Redundant Column

Removing the `promo_code_used` column because it duplicates the information stored in the discount variable.


In [ ]:
df=df.drop('promo_code_used', axis=1)

In [ ]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

### Save the Cleaned Dataset

Saving the cleaned dataset to Google Drive so it can be reused for further analysis in Power BI


In [ ]:
# Creating new file for cleaned data frame
new_file_path = '/content/drive/MyDrive/Data Analytics Project/customer_shopping_behavior_cleaned.csv'
df.to_csv(new_file_path, index=False)
print(f"Successfully saved to: {new_file_path}")

Successfully saved to: /content/drive/MyDrive/Data Analytics Project/customer_shopping_behavior_cleaned.csv


### Load Dataset into SQLite

Loading the cleaned dataset into an SQLite database so we can perform SQL queries and more advanced analytical operations.


In [21]:
import sqlite3

conn = sqlite3.connect("database.db")
file_path = '/content/drive/MyDrive/Data Analytics Project/customer_shopping_behavior_cleaned.csv'

df = pd.read_csv(file_path)
df.to_sql("customer_data", conn, if_exists="replace", index=False)


query = "SELECT * FROM customer_data LIMIT 10"

result = pd.read_sql(query, conn)
result





,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14.0
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14.0
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7.0
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7.0
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,NaN
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,2.9,Yes,Standard,Yes,14,Venmo,Weekly,Middle-aged,7.0
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,3.2,Yes,Free Shipping,Yes,49,Cash,Quarterly,Senior,90.0
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,3.2,Yes,Free Shipping,Yes,19,Credit Card,Weekly,Young Adult,7.0
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,2.6,Yes,Express,Yes,8,Venmo,Annually,Young Adult,NaN
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,4.8,Yes,2-Day Shipping,Yes,4,Cash,Quarterly,Middle-aged,90.0


### Revenue by Gender

Calculating the total revenue generated by each gender group to evaluate differences in purchasing behavior.

In [ ]:
query = """

SELECT gender,
SUM (purchase_amount) as revenue
FROM customer_data
GROUP BY gender;

"""

result = pd.read_sql(query, conn)
result


,gender,revenue
0,Female,75191
1,Male,157890


### High-Spending Customers Using Discounts

Identifying customers who used discounts but still spent more than the average purchase amount.  

This helps highlight valuable customers who respond to promotions while maintaining high spending levels.


In [22]:
query = """

SELECT
    customer_id,
    purchase_amount,
    item_purchased,
    category,
    location
FROM customer_data
WHERE discount_applied = 'Yes'
AND purchase_amount >= (SELECT AVG(purchase_amount) FROM customer_data)
ORDER BY purchase_amount DESC;

"""

result = pd.read_sql(query, conn)
result

,customer_id,purchase_amount,item_purchased,category,location
0,43,100,Coat,Outerwear,Tennessee
1,96,100,Sneakers,Footwear,Missouri
2,194,100,Belt,Accessories,North Dakota
3,205,100,Sneakers,Footwear,Arizona
4,244,100,Jewelry,Accessories,Kentucky
...,...,...,...,...,...
834,1247,60,Hoodie,Clothing,Louisiana
835,1296,60,Sneakers,Footwear,New Mexico
836,1333,60,Hoodie,Clothing,Alaska
837,1424,60,Sandals,Footwear,Michigan


### Highest Rated Products

Calculating the average review rating for each product and identify the five products with the highest ratings.  
This shows which products are most appreciated by customers, helping guide promotions, inventory, and recommendations.

In [ ]:
query = """

SELECT item_purchased, AVG(review_rating) AS avg_rating
FROM customer_data
GROUP BY item_purchased
ORDER BY avg_rating DESC
LIMIT 5;

"""
result = pd.read_sql(query, conn)
result

,item_purchased,avg_rating
0,Gloves,3.861429
1,Sandals,3.844375
2,Boots,3.818750
3,Hat,3.801299
4,Skirt,3.784810


### Spending by Shipping Method

*Comparing* the average purchase amount between customers who choose standard shipping and those who choose express shipping.

This highlights how shipping preference can signal willingness to spend, helping inform marketing and pricing strategies.


In [ ]:
query = """
SELECT ROUND(AVG(purchase_amount),2) AS avg_purchase_amount, shipping_type
FROM customer_data
WHERE shipping_type = 'Express'
OR
shipping_type = 'Standard'
GROUP BY shipping_type;


"""

result = pd.read_sql(query, conn)
result


,avg_purchase_amount,shipping_type
0,60.48,Express
1,58.46,Standard


### Spending by Subscription Status

Comparing spending behavior between subscribed and non-subscribed customers to evaluate the value of subscription programs.


In [ ]:
query = """
SELECT subscription_status,
COUNT(customer_id) AS totalt_customers,
ROUND(AVG(purchase_amount),2) AS avg_purchase_amount,
ROUND(SUM(purchase_amount),2) AS total_revenue
FROM customer_data
GROUP BY subscription_status;


"""

result = pd.read_sql(query, conn)
result

,subscription_status,totalt_customers,avg_purchase_amount,total_revenue
0,No,2847,59.87,170436.0
1,Yes,1053,59.49,62645.0


### Products Most Purchased with Discounts

Identifying the products that customers most frequently purchase when a discount is applied.  
This reveals which items are most sensitive to promotions, helping optimize discount strategies and boost sales.


In [ ]:
query = """

SELECT item_purchased,
ROUND(100 * SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END)/COUNT(*),2) as discount_rate
FROM customer_data
GROUP BY item_purchased
ORDER BY discount_rate DESC
LIMIT 5;

"""


result = pd.read_sql(query, conn)
result

,item_purchased,discount_rate
0,Hat,50.0
1,Sneakers,49.0
2,Coat,49.0
3,Sweater,48.0
4,Pants,47.0


### Customer Loyalty Segmentation

Segmenting customers into groups based on their purchase history to distinguish between new, returning, and loyal customers.  
This helps tailor marketing strategies, personalize offers, and strengthen customer retention.

In [ ]:
query = """

WITH customer_type AS (
SELECT customer_id, previous_purchases,
CASE
    WHEN previous_purchases = 1 THEN 'New'
    WHEN previous_purchases BETWEEN 2 AND 10 THEN 'Returning'
    ELSE 'Loyal'
    END AS customer_segment
FROM customer_data
)
select
customer_segment, COUNT(*) as "Number of Customers"
FROM customer_type
GROUP BY customer_segment;

"""


result = pd.read_sql(query, conn)
result

,customer_segment,Number of Customers
0,Loyal,3116
1,New,83
2,Returning,701


### Top Products by Category

Ranking products within each category based on purchase frequency to identify the most popular items.  
This helps highlight customer preferences within categories, guiding inventory, promotions, and merchandising decisions.


In [ ]:
query = """

WITH item_counts AS (
SELECT category,
       item_purchased,
       COUNT(customer_id) AS total_orders,
       ROW_NUMBER() OVER (PARTITION BY category ORDER BY COUNT(customer_id) DESC) AS item_rank
FROM customer_data
GROUP BY category, item_purchased
)

SELECT item_rank, category, item_purchased, total_orders
FROM item_counts
WHERE item_rank <= 3;



"""


result = pd.read_sql(query, conn)
result

,item_rank,category,item_purchased,total_orders
0,1,Accessories,Jewelry,171
1,2,Accessories,Sunglasses,161
2,3,Accessories,Belt,161
3,1,Clothing,Pants,171
4,2,Clothing,Blouse,171
5,3,Clothing,Shirt,169
6,1,Footwear,Sandals,160
7,2,Footwear,Shoes,150
8,3,Footwear,Sneakers,145
9,1,Outerwear,Jacket,163


### Relationship Between Repeat Buyers and Subscriptions

Analyzing whether loyal customers (repeat buyers) tend to be subscribed members.

Understanding this relationship helps evaluate the effectiveness of the subscription program. If repeat buyers are not heavily represented among subscribers, the business may benefit from strategies that convert loyal customers into subscription members.


In [ ]:
query= """

SELECT subscription_status, COUNT(customer_id) AS repeat_buyers
FROM customer_data
WHERE previous_purchases > 5
GROUP BY subscription_status

"""

result = pd.read_sql(query, conn)
result

,subscription_status,repeat_buyers
0,No,2518
1,Yes,958


### Revenue by Age Group

Calculating total revenue generated by each age group to identify which demographic segments contribute the most to overall sales.  
This helps target marketing efforts and tailor products/promotions to the highest-value customer segments.


In [ ]:
query= """

SELECT age_group,
SUM(purchase_amount) AS total_revenue
FROM customer_data
GROUP BY age_group
ORDER BY total_revenue DESC;

"""

result = pd.read_sql(query, conn)
result

,age_group,total_revenue
0,Young Adult,62143
1,Middle-aged,59197
2,Adult,55978
3,Senior,55763
